**Author:** Hui Fang\
**Purpose:** ST 554 Final Project\
Date: 4/20/2026

# Introction

This project demonstrates the end-to-end use of `PySpark` for both machine learning and real-time data processing. The data is adapted from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city), which examines how power consumption across different zones of Tetouan city relates to various factors such as time of day, temperature, and humidity.\
The project consistsof two components. The first component focuses on building an **Elastic Net regression model** using `Spark MLlib`, incorporating SQL-based feature engineering, one-hot encoding, PCA, and cross-validated hyperparameter tuning. The second component extends this work into a streaming context: a `Python` script generates incoming CSV files, `Spark` monitors the folder as a stream, applies the fitted model to produce predictions and residuals in real time, and outputs the results to the console.\
 Together, these components highlight Spark's ability to unify batch modeling and streaming inference within a single, conherent workflow.

# Goals
- create or use an already created github repo to document your work
    - You must commit often! At least 5 substantial commits must exist on this project. This allows us to see how your work develops over time. If this is not done, substantial credit will be lost.
- write a Jupyter notebook that fits a machine learning model using **pyspark's MLlib** module
- in that same notebook you'll write code to read in a stream of data (we'll produce that data ourselves using a **.py** file)
    - This **.py** file should also be included in the repo.
- we'll use the model to do predictions on the stream and write those out to the console!

Both the **.ipynb** file and the **.py** file should exist in your repo!

The data is modified from the UCI machine learning repository. The study was about relating power consumption from different zones of Tetouan city to various factors like time of day, temperature, and humidity.
- You'll have a chunk of the (modified) data to build your model on.
- You will then 'stream data' to a folder in the form of CSV files. You'll be monitoring this folder. When data comes in you'll use your fitted model to predict on the incoming data!

# Part I: Fitting Model
**Create a Jupyter notebook for the modeling fitting part and the Streaming part below.**
- The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv
- You should read this data into a standard pandas data frame using the **pd.read_csv()** function.
- Convert this to a spark data frame
- We are going to treat the **Power_Zone_3** variable as our response variable.
- We can use all of the other variables as predictors. (Imagine we know that the **Power_Zone_3** reading is going to go offline in the future and we need to be able to predict that value appropriately.)

We want to fit an elastic net model using CV (no training/test split, just using CV on the data we've read in) with the steps below.\
The transformations below should each use an **MLlib** function that can be put into a pipeline
- The Hour column is likely not stored as a DoubleType. If it is not, use an SQL transformer to cast the
variable as a DoubleType
- Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)
- One-hot encode the Month column
- Run a `PCA fit` on the **Temperature**, **Humidity**, **Wind_Speed**, **general_Diffuse_Flows**, and **Diffuse_Flows** columns.
    - To do this, I first used a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator.
    - Once fitted, then you'll have a PCA transformer we'll use in our pipeline.
    - We'll use two PCs in our transformation.
- Rename your response variable as label
- Use VectorAssembler() to put your predictors into a features. Use the
    -  two fitted PCA features
    - binary **Hour** variable
    - **Power_Zone_1**
    - **Power_Zone_2**
    - **Month** indicator variables
- This ends the pipeline of transformations!
- Now you'll then use the **CrossValidator()** function and the **LinearRegression()** function to fit an elastic net model.
    - You should do the following grid for the regParam and elasticNetParam: All combinations of\
    ***regParam:** 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1\
    ***elasticNetParam:** 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- Now fit the model using 5-fold CV with **rmse** as your criterion!
- Report the optimal values chosen for the tuning parameters
- Report the CV error
- Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and
evaluating on the entire training set
- Take the outputted transformations from the model (the predictions) and create a **residual** column (**label - prediction**). The **.withColumn()** method is handy here. Print out a data frame with these **residuals**, the **label** column, and the **predictions**.

First let's import the modules needed and create a Spark session.

In [2]:
# import modules needed
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml import Pipeline

# Create a Spark session
spark = SparkSession.builder.getOrCreate()

# prevent producing long list of warnings
spark.sparkContext.setLogLevel("ERROR")

## 1. Read in data and convert to Spark DataFrame

In [3]:
# read in data
power_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_pd.head()
# convert to Spark Dataframe
power_sdf = spark.createDataFrame(power_pd)
# print out the data frame schema
power_sdf.printSchema()
# check the spark DataFrame
power_sdf.show(10)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

## 2. Cast `Hour` to Double using SQLTransformer

The printed schema indicates that `Hour` is stored as a long. To ensure compatibility with downstream `MLlib` transformations, I cast `Hour` to `DoubleType` using an `SQLTransformer` within the pipeline.

In [4]:
# cast Hour vaiable to double
hour_cast_sql = SQLTransformer(statement = "SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

## 3. Binarize the `Hour` column and One-hot encode the `Month` column

Next, I create a binary indicator for the `Hour` variable, where values less than 6.5 represent nighttime and values greater than or equal to 6.5 represent daytime. Since `Month` is already stored as a numeric type, we can directly apply OneHotEncoder without any additional casting.

In [5]:
# use Binarizer on Hour_double
hour_bin = Binarizer(
    inputCol = "Hour_double",
    outputCol = "Hour_binary",
    threshold = 6.5)

# One-hot encode the month variable
month_encoder = OneHotEncoder(
    inputCols = ["Month"],
    outputCols = ["Month_ohe"])

## 4. Fit a PCA Model

To fit a PCA (Principal Component Analysis), which reduces several correlated variables into a smaller set of summary components, I first assemble `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` into a single feature vector using `VectorAssembler()`. After  fitting the PCA model, Spark produces a PCA transformer that becomes part of the pipeline. In this project, I retain two principal components (PCs).

In [6]:
# assemble variables into a single vector
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows","Diffuse_Flows"],
    outputCol = "pca_input")

# fit a PCA model
pca = PCA(
    k = 2,
    inputCol = "pca_input",
    outputCol = "pca_features")

## 5. Rename response to `label` and assemble the final `features` vector

Next, I add an `SQLTransformer` to rename `Power_Zone_3` as `label`. I then use a final `VectorAssembler` to combine predictors `Hour_binary`, `Power_Zone_1`, `Power_Zone_2`, the one-hot encoded `Month_ohe`, and the two PCA compoents (`pca_features`) into a gingle `features` column for model fitting.

In [16]:
# rename sesponse `Power_Zone_3` to `label`
label_sql = SQLTransformer(
    statement = """
        SELECT *, Power_Zone_3 AS label
        FROM __THIS__
    """
)

# combine preditors into `features`
features_assembler = VectorAssembler(
    inputCols = [
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_ohe",
        "pca_features"
    ],
    outputCol = "features")

## 6. Build the transformation pipeline

Now I chain all transformation stages together into a single Spark MLlib pipeline, ensuring that each transformation is applied in the correct order before model fitting.

In [18]:
# build the pipeline
transform_stages = [
    hour_cast_sql,
    hour_bin,
    month_encoder,
    label_sql,
    pca_assembler,
    pca,
    features_assembler]

transform_pipeline = Pipeline(stages = transform_stages)

# fit the transformation pipeline
transform_model = transform_pipeline.fit(power_sdf)

# transformed data with label + features
train_transformed = transform_model.transform(power_sdf)
train_transformed.select("label", "features").show(5, truncate = False)

+-----------+---------------------------------------------------------------------------------------+
|label      |features                                                                               |
+-----------+---------------------------------------------------------------------------------------+
|20240.96386|(17,[1,2,4,15,16],[34055.6962,16128.87538,1.0,1.7944048636569532,-0.7412746447404528]) |
|20131.08434|(17,[1,2,4,15,16],[29814.68354,19375.07599,1.0,1.8060408300982302,-0.7108534239558553])|
|19668.43373|(17,[1,2,4,15,16],[29128.10127,19006.68693,1.0,1.8102297630563893,-0.728311319115901]) |
|18899.27711|(17,[1,2,4,15,16],[28228.86076,18361.09422,1.0,1.7986676517408833,-0.7220913032200016])|
|18442.40964|(17,[1,2,4,15,16],[27335.6962,17872.34043,1.0,1.86328720163797,-0.7323046647776632])   |
+-----------+---------------------------------------------------------------------------------------+
only showing top 5 rows


The output confirms that the transformation pipeline executed correctly: the response variable has been renamed to `label`, and all predictors have been successfully assembled into the unified `features` vector required for model fitting.

## 7. Fit the Elastic Net model

Once the transformation pipeline has produced the final `label` and `features` columns, I proceed to fit the `Elastic Net` regression model. I begin by importing the required modules.

In [19]:
# import modules needed
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml.tuning import CrossValidator

After completing the transformation pipeline, we proceed to fit the `Elastic Net` regression model. This involves instantiating the LinearRegression estimator, constructing the hyperparameter grid for both regularization strength and Elastic Net mixing, defining the RMSE evaluator, configuring the 5-fold CrossValidator, and finally fitting the cross-validated model on the transformed dataset.

In [20]:
# create a regression model instance
lr = LinearRegression(
    featuresCol = "features", 
    labelCol = "label",
    maxIter=5000,
    solver="auto")

# build the parameter grid
paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .build()
)

# define the evaluator
evaluator = RegressionEvaluator(
    predictionCol = "prediction",
    labelCol = "label",
    metricName = "rmse")

# set up the CrossValidator
cv = CrossValidator(
    estimator = lr,
    estimatorParamMaps = paramGrid,
    evaluator = evaluator,
    numFolds = 5,                   # set 5-fold CV
    parallelism = 8,                # run up to 8 CV stasks simultaneously
    seed = 42                       # set seed for reproducibility
)

Now, I fit the cross-validated `Elastic Net` model on the transformed dataset. This step trains all 121 hyperparameter combinations, evaluates each using 5-fold cross-validation, and automatically selects the model with the lowest RMSE. Because the grid is large, it may take several mintues to fit the CV model.

In [21]:
# fit the CV model on transformed data
cv_model = cv.fit(train_transformed)

26/04/27 22:49:34 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/04/27 22:50:45 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed


After fitting the model, I extract the best hyperparameters to inspect the best `Elastic Net` model and report the optimal tuning parameters.

In [22]:
# extract the best model
best_model = cv_model.bestModel

# extract the best tuning parameters
best_reg = best_model._java_obj.getRegParam()
best_alpha = best_model._java_obj.getElasticNetParam()

# print out the best tunning parameters
print("Best regParam (λ):", best_reg)
print("Best elasticNetParam (α):", best_alpha)

# report the CV error (RMSE)
best_cv_rmse = min(cv_model.avgMetrics)
print("Best CV RMSE:", best_cv_rmse)

Best regParam (λ): 0.05
Best elasticNetParam (α): 0.1
Best CV RMSE: 2147.5897872581236


The reusults show that the optimal tuning parameters selected by cross-validation are λ = 0.1 and α = 0.25, with the corresponding minimum RMSE of 2147.70.\
Using these best-performing hyperparameters, I can now generate predictions with the fitted Elastic Net model.

In [23]:
# make prediction
train_predictions = best_model.transform(train_transformed)

# extract and print out training RMSE
train_rmse = evaluator.evaluate(train_predictions)
print("Training RMSE:", train_rmse)

Training RMSE: 2147.0973169293943


The trarining set RMSE is 2147.10, which is very close to but slightly lower than the cross-validated RMSE. The small difference indicates that the selected `Elastic Net` model generalizes well and does not exhibit overfitting.

Next, I compute the residuals for the fitted model and present a DataFrame that includes the true `label`, the model's prediction, and the residual (defined as `label - prediction`).

In [24]:
# import module needed
from pyspark.sql.functions import col

# create residual data frame
residuals_df = (
    train_predictions
    .withColumn("residual", col("label") - col("prediction"))
    .select("label", "prediction", "residual")
)
# show the first 10 rows of the results
residuals_df.show(10, truncate = False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20878.85066078908 |-637.8868007890815|
|20131.08434|18660.227266543898|1470.8570734561035|
|19668.43373|18204.7521531144  |1463.6815768855995|
|18899.27711|17590.64849833898 |1308.6286116610208|
|18442.40964|16997.301986456692|1445.1076535433094|
|18130.12048|16517.6867234941  |1612.4337565059031|
|17945.06024|16093.246141053285|1851.8140989467138|
|17459.27711|15722.695360253703|1736.5817497462958|
|17025.54217|15271.043828662601|1754.4983413373993|
|16794.21687|14938.348764665414|1855.8681053345863|
+-----------+------------------+------------------+
only showing top 10 rows


# Part II: Streaming

For Part II, I construct a streaming workflow by reading .csv files as they arrive in a designated input directory. I generate these files by randomly sampling rows from the full dataset and writing them out sequentially using a python script, thereby emulating a real-time data stream. The original dataset can be accessed [here](https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv).

## 1. Read in data and obtain the schema
- We're going to read in a stream in the form of .csv files. Create a folder where you will be sending
your .csv files.
- Setup the schema for the stream (you can use the schema from the original data as we did in hw 10)
- Set up the readStream. Be sure to add header = True as you'll likely be outputting files with a header and we don't need to read that in.

I load the full dataset once in batch mode to inspect the structure and extract a schema. Spark requires an explicit schema for Structured Streaming because it cannot infer column types from streaming files. This schema is then applied to all incoming CSV files generated by the Python script.

In [25]:
# import modules needed
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA

# Create a Spark session
spark = SparkSession.builder.getOrCreate()

# prevent producing long list of warnings
spark.sparkContext.setLogLevel("ERROR")

Python code to generate sample files

```
import os
import time
import pandas as pd

# directory of this script
script_dir = os.path.dirname(os.path.abspath(__file__))

# full path to stream_input inside the project folder
input_dir = os.path.join(script_dir, "stream_input")

# 1. Ensure the input directory exists
os.makedirs(input_dir, exist_ok=True)

# 2. Clear existing files in the stream_input directory
for f in os.listdir(input_dir):
    path = os.path.join(input_dir, f)
    if os.path.isfile(path):
        os.remove(path)

# 3. Load the full streaming dataset
csv_path = os.path.join(script_dir, "power_streaming_data.csv")
df = pd.read_csv(csv_path)

print("Starting data generation...")

# 4. randomly sample new files
for i in range(20):
    sample = df.sample(n = 5)                               # sample 5 rows
    out_path = os.path.join(input_dir, f"sample_{i}.csv")   # build the full file path for the i‑th sample inside stream_input
    sample.to_csv(out_path, index = False)                  # write the sampled file to input folder
    print(f"Wrote: {out_path}")                             # print the full path of the file that was just created
    time.sleep(10)                                          # pause 10 seconds between outputs

print("Data generation completed")
```

## 2. Set up the schema and the streaming source
First, I want to ensure that the streaming input directory exists and then extract the schema from the batch DataFrame. I create the streaming input directory (if needed), extract the schema from the batch DataFrame, and configure Spark to continuously monitor the folder for new CSV files.

In [31]:
# extract the schema from the static Spark DataFrame
schema = power_sdf.schema

# turn the folder into a live data source
stream_df = (
    spark.readStream
        .schema(schema)            # apply the predefined schema
        .option("header", True)    # ensure incoming files include a header row
        .csv("stream_input"))      # folder where streaming files will appear

## 3. Transform the streaming data and generate predictions
- Now, we'll do two separate things on the stream and join them together:
    - With your stream, use your model transformer to obtain predictions from the incoming data. On the resulting predictions also create a **residual** column as noted in the previous section (return only the **label**, **prediction** and **residual** columns from this part)
    - We can use our stream more than once! With another transformation on the (original) stream, modify the response variable to be called **label**.
    - Now join your above transform with this stream based on the **label** variable which should be common to both!
    - Note 1: This is a little silly, but I want you to join two transformations of the stream and I don't want things to get too crazy
    - Note 2: Each data frame is created from the same stream of data! You don't need two streams, you can use the same stream and just do two separate transformations on it, combining it with a **.join()** method from one of the SQL style data frames you are dealing with (as we discussed in the notes)


In this step, I process the incoming data stream to prepare it for model prediction and then generate predictions and residuals.
First, I rename the original response variable in the raw `stream_df` to `label` (stored as `label_df`), which will be used for joining with predictions later.
Next, I apply the *entire fitted transformation pipeline* from Part-I to the `stream_df` (creating `stream_transformed`). This ensures all feature engineering steps are consistently applied, preparing the data with the same `features` vector used during training.\
Finally, I apply the trained Elastic Net regression `cv_model` to the `stream_transformed` data to obtain predictions. I then compute residuals (the difference between the observed `label` and the `prediction`) and retain only the `label`, `prediction`, and `residual` columns in the `pred_df` for output.

In [28]:
# import modules needed
import pyspark
from pyspark.sql.functions import col

# build a streaming transformer
stream_transformer = transform_model

# rename the response variable to 'label'
label_df = stream_df.withColumnRenamed("Power_Zone_3", "label")

# apply the streaming transformer
stream_transformed = stream_transformer.transform(stream_df)

# apply the trained Elastic Net regression model to obtain predictions on the streaming data
pred_df = cv_model.transform(stream_transformed)

# compute residuals for each incoming record
pred_df = (pred_df
           .withColumn("residual", col("label") - col("prediction"))
           .select("label", "prediction", "residual") # only keep the target columns
        )

# join the prediction DataFrame with the renamed raw stream on the 'label'
joined_df = pred_df.join(label_df, on = "label")

# check the schema of joined data frame
joined_df.printSchema()

root
 |-- label: double (nullable = true)
 |-- prediction: double (nullable = false)
 |-- residual: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



## 4. Start the streaming query to produce data
- Now write your stream to the console using the append output mode.
- Start the query!

You should have the file we'll use for streaming data downloaded and in a place you can locate. Create a .py file that reads that into a pandas (regular) data frame and does the following.
- Writes a loop (say 20 iterations) to:
    - Randomly sample five rows and output those to a .csv file in the folder you are watching with your stream.
    - Be sure not to write out the indices. You can leave the column names as long as you handle that on your stream appropriately
    - Pause for 10 seconds in between outputting of data sets
    - Submit this loop in a python console.
- While the loop runs, you should see output in your notebook!




Finally, I configure the streaming sink. Using append mode, the console sink prints each processed micro-batch to the console as new files arrive in the input directory while the Python script is running.

In [39]:
# start the query and write output to the console in append mode
query = (
    joined_df.writeStream
             .outputMode("append")                 # append new rows as they arrive
             .format("console")                    # print results to the console
             .option("truncate", False)            # print all columns
             .option("numRows", 5)                 # print only the first 5 rows
             .trigger(processingTime = "1 second") # add a trigger
             .start()                              # start the streaming query
)

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|18780.55385|20323.23288132605 |-1542.6790313260499|28.19      |32.86   |4.918     |909.0                |52.08        |36874.17219 |23134.30353 |6    |12  |
|9494.954407|10046.153290401591|-551.198883401592  |17.2       |78.0    |0.083     |7.12                 |5.936        |28384.07002 |17925.3112  |10   |7   |
|25891.08434|24953.936509255796|937.1478307442048  |14.99      |74.8    |0.071     |0.07                 |0.115  

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15983.27638|16979.10477176048 |-995.8283917604822 |10.51      |87.0    |4.92      |577.6                |323.9        |33742.37288 |20221.2766  |2    |11  |
|15832.76382|15703.7260797235  |129.0377402765007  |16.15      |50.9    |0.085     |525.9                |41.83        |30648.81356 |17244.9848  |2    |11  |
|18832.34043|19932.951862334965|-1100.6114323349648|21.29      |65.86   |4.923     |0.102                |0.056  

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13889.03226|13226.882688917452|662.1495710825475  |12.82      |86.1    |0.089     |0.018                |0.174        |22666.21277 |14008.53659 |3    |4   |
|15744.0    |15624.78657745308 |119.2134225469199  |11.69      |83.3    |0.081     |0.037                |0.111        |23957.63186 |13644.80652 |4    |5   |
|20096.80251|22484.69534349464 |-2387.8928334946395|26.02      |39.58   |4.905     |0.102                |0.07   

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16376.92462|16053.045571103394|323.87904889660604 |8.78       |87.8    |0.084     |0.059                |0.111        |26347.11864 |16559.27052 |2    |0   |
|18996.36364|19698.14872102326 |-701.7850810232594 |16.52      |74.3    |0.071     |270.4                |247.0        |34510.39828 |20507.53564 |4    |13  |
|11178.79518|10027.579940034884|1151.215239965115  |4.509      |74.5    |0.084     |6.643                |6.494  

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual          |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25727.61134|26488.429487046782|-760.8181470467825|17.66      |84.0    |4.922     |0.062                |0.096        |45009.83607 |27406.81115 |5    |22  |
|12926.44377|10717.570033703774|2208.8737362962256|23.47      |61.09   |4.916     |0.091                |0.104        |26858.99344 |16121.57676 |10   |2   |
|11357.41935|9739.786060829345 |1617.6332891706552|11.84      |71.0    |0.085     |124.0                |116.0        

...

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual          |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14507.09548|12916.472000048998|1590.623479951002 |13.77      |73.0    |0.07      |0.077                |0.145        |21825.76271 |12868.08511 |2    |3   |
|17881.44578|17470.014124786143|411.43165521385527|5.611      |73.7    |0.085     |0.062                |0.1          |28095.18987 |17992.70517 |1    |0   |
|20288.25911|18548.570394950104|1739.688715049895 |22.03      |64.36   |4.917     |895.0                |56.4        

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual          |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25251.37688|24612.282787848148|639.0940921518522 |9.64       |86.0    |4.917     |0.066                |0.163        |41882.0339  |25010.33435 |2    |19  |
|15886.26506|18912.24282962437 |-3025.977769624371|15.97      |78.3    |0.07      |199.1                |201.2        |34432.40506 |21457.75076 |1    |11  |
|18099.41423|19376.131643301695|-1276.717413301696|22.78      |63.84   |4.92      |403.4                |365.7       

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|8794.650456|7045.238177189352 |1749.412278810647  |18.74      |85.0    |0.066     |0.128                |0.089        |23229.05908 |18746.88797 |10   |7   |
|24154.64435|24006.063804515597|148.58054548440123 |26.6       |59.11   |4.919     |821.0                |127.8        |32754.81728 |19435.44304 |7    |12  |
|13359.03614|13649.521102050978|-290.48496205097763|11.55      |71.1    |0.076     |0.091                |0.134 

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15903.24821|16146.454952558328|-243.2067425583282 |26.24      |46.62   |0.323     |157.4                |179.5        |36331.32743 |22412.05821 |9    |18  |
|26365.85774|28857.294599450874|-2491.436859450874 |27.95      |65.18   |4.921     |851.0                |99.7         |39471.62791 |26570.88608 |7    |14  |
|18216.72727|18933.688054740556|-716.9607847405568 |15.46      |82.8    |0.072     |96.6                 |89.8  

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|29627.07692|30151.03239175658 |-523.9554717565807 |23.98      |46.66   |4.919     |3.097                |2.679        |48953.64238 |26775.46778 |6    |20  |
|13434.16413|13501.318426266975|-67.1542962669755  |27.36      |49.55   |4.926     |438.3                |116.0        |35038.94967 |24273.85892 |10   |15  |
|9991.836735|11405.242245271238|-1413.4055102712373|13.55      |59.93   |0.086     |499.4                |37.36 

In [40]:
# stop the query
query.stop()

## 5. To Submit

Submit a URL to your github repo on Moodle! Upload the video on Moodle as well! (Both on the assignment link.)

Create a short (one to three minute) video that shows you start the query, start the loop, and then watching the pyspark output update. This can easily be done with zoom. Please don't make the video very long as I want you to upload it to Moodle! Note: If you don't include the video you will lose substantial
credit.